# Week 8 Lab — Regularization with Hitters

**DATA 201 · Rebin Muhammad · Montgomery College**

Today we predict baseball player salary with ordinary least squares, ridge, and lasso. Salary is continuous, so we use **RMSE** and **R²**, not accuracy or ROC-AUC.

### Learning goals
By the end of this lab you should be able to:
1. Fit a linear regression baseline and spot overfitting with train vs. test error.
2. Explain how ridge (L2) and lasso (L1) change the fitted coefficients.
3. Choose `alpha` with **5-fold cross-validation on the training set only**.
4. Compare models fairly on an untouched test set.
5. Discuss the tradeoff among bias, variance, accuracy, and interpretability.

**Connection to earlier weeks:** Week 7 used regularization with logistic regression. Here the outcome is continuous, so the same idea (penalize large coefficients) appears in linear regression.


### Part 1 — Load the Hitters data

The ISLP Hitters data set records 1986 season statistics and 1987 salaries for major-league players. `Salary` is measured in **thousands of dollars**.

Some players have a missing salary. We will drop those rows before modeling.

**Colab note:** Run the next cell once per session to install ISLP.


In [ ]:
%pip -q install ISLP


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ISLP import load_data
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
pd.set_option("display.max_columns", 30)


In [ ]:
hitters = load_data("Hitters")
print("Original shape:", hitters.shape)
display(hitters.head())
print("\nMissing values by column:")
print(hitters.isna().sum().sort_values(ascending=False).head(10))


In [ ]:
# Salary is recorded in thousands of dollars
print("Salary range (thousands of dollars):")
print("  min =", hitters["Salary"].min())
print("  max =", hitters["Salary"].max())
print(
    "In dollars, that is about",
    int(hitters["Salary"].min() * 1000),
    "to",
    int(hitters["Salary"].max() * 1000),
)


**Discuss before you code**
- Which column is the outcome?
- Which columns look categorical?
- Why must we remove players with a missing salary before we train and evaluate a model?

Only `Salary` has missing values here. After we drop those rows, we will encode the categorical predictors **inside a pipeline**, so the encoding is fit on training data only during cross-validation.


In [ ]:
# Keep only rows with a known salary
hitters = hitters.dropna(subset=["Salary"]).copy()

# Predictors and target
X = hitters.drop(columns="Salary")
y = hitters["Salary"]  # thousands of dollars

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
category_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric predictors:", len(numeric_cols))
print("Categorical predictors:", category_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Rows: {len(X)} total | {len(X_train)} train | {len(X_test)} test")


### Part 2 — Preprocess and fit a baseline

**Why scale?** Ridge and lasso penalize the **size of coefficients**. Career at-bats and years played live on different scales. Without standardizing the numeric predictors, the penalty treats those columns unfairly.

We use the **same preprocessing** for ordinary least squares, ridge, and lasso. Putting the scaler and encoder inside a `Pipeline` keeps each cross-validation fold honest: the scaler is fit only on that fold's training rows.


In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), category_cols),
        ]
    )


def measure(name, fitted_model):
    # Train and test scores for a fitted pipeline
    train_pred = fitted_model.predict(X_train)
    test_pred = fitted_model.predict(X_test)
    return {
        "Model": name,
        "Train RMSE": np.sqrt(mean_squared_error(y_train, train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Train R²": r2_score(y_train, train_pred),
        "Test R²": r2_score(y_test, test_pred),
    }


baseline = Pipeline([
    ("preprocess", make_preprocessor()),
    ("model", LinearRegression()),
])
baseline.fit(X_train, y_train)

results = [measure("Linear regression (no penalty)", baseline)]
display(pd.DataFrame(results).round(2))


**How to read the table**
- **RMSE** is in thousands of dollars (same units as `Salary`). Lower is better.
- **R²** is the share of salary variation explained by the model. Higher is better.
- Training error is usually a little better than test error. A **large** gap can signal overfitting. A small gap does not, by itself, prove the model is excellent.

**Question 1.** Report the baseline test RMSE. Why is training RMSE often smaller than test RMSE?


### Part 3 — Turn the regularization dial

Ordinary least squares minimizes training RSS only.

Ridge and lasso still care about training fit, but they also pay a penalty for large coefficients:

- **Ridge (L2):** \(\mathrm{RSS} + \alpha \sum_j w_j^2\). Shrinks coefficients toward zero. Usually keeps every predictor.
- **Lasso (L1):** \(\mathrm{RSS} + \alpha \sum_j |w_j|\). Can set some coefficients exactly to zero (automatic feature selection).

ISLP writes the penalty strength as \(\lambda\). scikit-learn calls the same idea `alpha` for `Ridge` and `Lasso`. Larger `alpha` means a stronger penalty. The intercept is not penalized.

In this part we fix the training set and change `alpha`, just to see what happens. In Part 4 we will let cross-validation choose `alpha`.


In [ ]:
alpha_examples = [0.01, 0.1, 1, 10, 100, 1000]
rows = []

for kind, Estimator in [("Ridge", Ridge), ("Lasso", Lasso)]:
    for a in alpha_examples:
        model = Estimator(alpha=a, max_iter=30000) if kind == "Lasso" else Estimator(alpha=a)
        pipe = Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", model),
        ])
        pipe.fit(X_train, y_train)
        coefs = pipe.named_steps["model"].coef_

        rows.append({
            "Model": kind,
            "alpha": a,
            "Nonzero coefficients": int(np.sum(np.abs(coefs) > 1e-6)),
            "Sum of |coefficients|": float(np.sum(np.abs(coefs))),
            "Training RMSE": np.sqrt(mean_squared_error(y_train, pipe.predict(X_train))),
        })

alpha_table = pd.DataFrame(rows)
display(alpha_table.round(2))


**What to look for**
- **Ridge:** coefficients shrink as `alpha` grows, but predictors typically stay in the model.
- **Lasso:** some coefficients become exactly zero, so the model becomes sparse.
- **Training RMSE:** often rises as regularization gets stronger, because the model has less freedom to fit the training sample.

The “sum of |coefficients|” column is only a summary of size. Ridge still uses an L2 penalty during fitting; lasso uses L1.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, kind in zip(axes, ["Ridge", "Lasso"]):
    subset = alpha_table[alpha_table["Model"] == kind]
    ax.semilogx(subset["alpha"], subset["Nonzero coefficients"], marker="o")
    ax.set_title(f"{kind}: predictors kept as alpha increases")
    ax.set_xlabel("alpha (log scale)")
    ax.set_ylabel("Number of nonzero coefficients")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


**Question 2.** How does the number of nonzero coefficients change for ridge and for lasso? What happens to training RMSE as \(\alpha\) increases? Why does a higher training error **not necessarily** mean a worse model for new data?


### Part 4 — Choose `alpha` with cross-validation

Do **not** pick `alpha` because it looks best on the test set. The test set is reserved for the final comparison.

Workflow (same idea as Weeks 6–7):

**Pipeline → GridSearchCV → 5-fold CV on training data → choose alpha → refit on all training data → score the test set once.**

We use `scoring="neg_mean_squared_error"` because `GridSearchCV` maximizes its score. Negative MSE turns “smaller error is better” into “larger score is better.”


In [ ]:
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_grid = GridSearchCV(
    Pipeline([("preprocess", make_preprocessor()), ("model", Ridge())]),
    {"model__alpha": np.logspace(-2, 5, 40)},
    scoring="neg_mean_squared_error",
    cv=cv5,
    n_jobs=-1,
)

lasso_grid = GridSearchCV(
    Pipeline([("preprocess", make_preprocessor()), ("model", Lasso(max_iter=30000))]),
    {"model__alpha": np.logspace(-2, 4, 40)},
    scoring="neg_mean_squared_error",
    cv=cv5,
    n_jobs=-1,
)

ridge_grid.fit(X_train, y_train)
lasso_grid.fit(X_train, y_train)

print("Best ridge alpha:", ridge_grid.best_params_["model__alpha"])
print("Best ridge CV RMSE:", np.sqrt(-ridge_grid.best_score_))
print("Best lasso alpha:", lasso_grid.best_params_["model__alpha"])
print("Best lasso CV RMSE:", np.sqrt(-lasso_grid.best_score_))


Each grid tries many values of `alpha` across 5 folds, then refits the winning pipeline on all of `X_train`.

`n_jobs=-1` uses available CPU cores so the grid search finishes faster. In Colab, that work runs on Colab’s machines, not on your laptop.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for label, grid in [("Ridge (L2)", ridge_grid), ("Lasso (L1)", lasso_grid)]:
    alphas = grid.cv_results_["param_model__alpha"].astype(float)
    cv_rmse = np.sqrt(-grid.cv_results_["mean_test_score"])
    ax.semilogx(alphas, cv_rmse, marker=".", label=label)

ax.set_xlabel("alpha (log scale)")
ax.set_ylabel("5-fold CV RMSE (salary, thousands of dollars)")
ax.set_title("Choose alpha from validation results, not the test set")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


**Question 3.** For each model, report the best \(\alpha\) and the CV RMSE. What happens when \(\alpha\) becomes **too large**? Use the word **bias** in your answer.

As \(\alpha\) grows, coefficients are pushed toward zero. If the penalty is too strong, the model becomes too simple: bias rises, variance falls, and both training and validation error can get worse. Cross-validation tries to stop before that point.


### Part 5 — Final comparison on the untouched test set

`GridSearchCV` already refit each winning pipeline on all training rows (`refit=True`). Now compare ordinary least squares, tuned ridge, and tuned lasso on the **same** test rows.


In [ ]:
results += [
    measure("Ridge (L2, tuned)", ridge_grid.best_estimator_),
    measure("Lasso (L1, tuned)", lasso_grid.best_estimator_),
]

comparison = pd.DataFrame(results).sort_values("Test RMSE")
display(comparison.round(2))


In [ ]:
ax = comparison.plot(
    kind="bar",
    x="Model",
    y="Test RMSE",
    legend=False,
    figsize=(8, 4),
    color=["#0072b2", "#0f7a72", "#c45c26"],
)
ax.set_ylabel("Test RMSE (salary, thousands of dollars)")
ax.set_title("Same test rows, three models")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


**Question 4.** Which model has the lowest test RMSE? Are the differences large enough that you would confidently call one model clearly better? Remember: one split is useful evidence, not proof that a small gap will always hold.

Regularization trades a little bias for lower variance. Whether that improves test RMSE depends on the data.


### Part 6 — Interpret the coefficients

The table below compares coefficients for the same transformed features under OLS, tuned ridge, and tuned lasso. Numeric predictors were standardized and categorical predictors were one-hot encoded, so the three columns are comparable **feature by feature**. Still be careful when ranking “importance” across very different kinds of predictors.


In [ ]:
feature_names = baseline.named_steps["preprocess"].get_feature_names_out()

coef_table = pd.DataFrame({
    "Feature": feature_names,
    "OLS": baseline.named_steps["model"].coef_,
    "Ridge": ridge_grid.best_estimator_.named_steps["model"].coef_,
    "Lasso": lasso_grid.best_estimator_.named_steps["model"].coef_,
}).set_index("Feature")

display(coef_table.round(2))

lasso_keep = int(np.sum(np.abs(coef_table["Lasso"]) > 1e-6))
print(f"Lasso kept {lasso_keep} of {len(coef_table)} coefficients nonzero.")


In [ ]:
coef_table[["OLS", "Ridge", "Lasso"]].plot(kind="barh", figsize=(10, 8))
plt.axvline(0, color="#1c2430", linewidth=1)
plt.xlabel("Fitted coefficient (salary in thousands of dollars)")
plt.title("Coefficients: unpenalized vs. tuned ridge and lasso")
plt.tight_layout()
plt.show()


**Question 5.** Name two predictors retained by lasso and two that were set to zero (if available). Does a zero coefficient prove that variable has no relationship with salary? Why do **correlated predictors** make that claim especially risky?

Career hits, career runs, and career at-bats often move together. Overlap like that can make OLS coefficients unstable. Ridge tends to shrink a correlated group together. Lasso may keep one member of the group and drop the others.

**Interpretability vs. flexibility:** Lasso can give a shorter story (fewer features). Ridge usually keeps more predictors. Neither method turns a linear model into a nonlinear one by itself; both constrain coefficients and therefore change effective complexity.


### Exit ticket (3–5 sentences)

A student says:

> “Lasso removed five predictors, so those predictors do not affect salary. Also, ridge has a higher training error than linear regression, so ridge is clearly worse.”

What is wrong with **both** claims? Use training vs. test performance, bias–variance, and interpretability in your answer.
